# ModernBERT Edge Scoring

Colab notebook để tải P5 transitive-pruned output từ Google Drive, chạy `answerdotai/ModernBERT-base`, và xuất điểm `edge_strength` / `bidirectional_score` cho các `clean_candidate_edges`.

Input mặc định là Drive link bạn đã upload. Output tải về là `modernbert_edge_scores.json`.

In [1]:
DRIVE_URL = "https://drive.google.com/file/d/1HyWTp9bJv2lg1EF17H_vxVSZnYLY4lDm/view?usp=sharing"
MODEL_ID = "answerdotai/ModernBERT-base"
MAX_LENGTH = 512
BATCH_SIZE = 16
OUTPUT_PATH = "/content/modernbert_edge_scores.json"

In [2]:
!pip -q install -U "transformers>=4.57.0" accelerate gdown safetensors

In [3]:
import json
import math
import re
import shutil
import requests
import zipfile
from pathlib import Path

download_path = Path("/content/p5_drive_input")
extract_dir = Path("/content/p5_drive_extract")
if download_path.exists():
    download_path.unlink()
if extract_dir.exists():
    shutil.rmtree(extract_dir)

def extract_drive_file_id(url: str) -> str:
    patterns = [r"/file/d/([A-Za-z0-9_-]+)", r"[?&]id=([A-Za-z0-9_-]+)"]
    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)
    raise ValueError(f"Cannot extract Google Drive file id from: {url}")

def download_from_drive(file_id: str, output_path: Path) -> None:
    import gdown

    result = gdown.download(id=file_id, output=str(output_path), quiet=False)
    if result and output_path.exists() and output_path.stat().st_size > 0:
        return

    # Fallback for environments where gdown CLI/API rejects fuzzy URLs.
    session = requests.Session()
    url = "https://drive.google.com/uc?export=download"
    response = session.get(url, params={"id": file_id}, stream=True)
    token = None
    for key, value in response.cookies.items():
        if key.startswith("download_warning"):
            token = value
            break
    if token:
        response = session.get(url, params={"id": file_id, "confirm": token}, stream=True)
    response.raise_for_status()
    with output_path.open("wb") as handle:
        for chunk in response.iter_content(1024 * 1024):
            if chunk:
                handle.write(chunk)
    if not output_path.exists() or output_path.stat().st_size == 0:
        raise RuntimeError("Google Drive download produced an empty file. Check that the file is shared publicly or use Colab file upload.")

drive_file_id = extract_drive_file_id(DRIVE_URL)
print("Drive file id:", drive_file_id)
download_from_drive(drive_file_id, download_path)
print("Downloaded:", download_path, "bytes=", download_path.stat().st_size)

if zipfile.is_zipfile(download_path):
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(download_path) as zf:
        zf.extractall(extract_dir)
    json_files = sorted(extract_dir.rglob("*.json"))
    if not json_files:
        raise FileNotFoundError("Zip downloaded from Drive has no .json files")
    input_json_path = json_files[0]
else:
    input_json_path = download_path

payload = json.loads(input_json_path.read_text(encoding="utf-8"))
print("Input JSON:", input_json_path)
print("Top-level keys:", list(payload.keys()))

Drive file id: 1HyWTp9bJv2lg1EF17H_vxVSZnYLY4lDm


Downloading...
From: https://drive.google.com/uc?id=1HyWTp9bJv2lg1EF17H_vxVSZnYLY4lDm
To: /content/p5_drive_input
100%|██████████| 169k/169k [00:00<00:00, 3.80MB/s]

Downloaded: /content/p5_drive_input bytes= 169261
Input JSON: /content/p5_drive_input
Top-level keys: ['run_id', 'stage_id', 'clean_candidate_edges', 'pruned_edges', 'adjudication_trace', 'transitive_prune_summary']


In [4]:
clean_edges = payload.get("clean_candidate_edges")
if not isinstance(clean_edges, list):
    raise ValueError("Expected top-level clean_candidate_edges[] in P5 transitive-pruned output")

global_kps = payload.get("global_kps") or payload.get("concepts_kp_global") or []
kp_index = {}
for kp in global_kps:
    if isinstance(kp, dict):
        kp_id = kp.get("global_kp_id") or kp.get("kp_id")
        if kp_id:
            kp_index[kp_id] = kp

def readable_kp_id(kp_id: str) -> str:
    return re.sub(r"[_-]+", " ", kp_id.removeprefix("kp_")).strip()

def kp_text(kp_id: str) -> str:
    kp = kp_index.get(kp_id, {})
    name = kp.get("name") or kp.get("canonical_name") or readable_kp_id(kp_id)
    desc = kp.get("description") or kp.get("global_description") or ""
    role = kp.get("structural_role") or ""
    importance = kp.get("importance_level") or ""
    parts = [f"Concept: {name}."]
    if desc:
        parts.append(f"Description: {desc}")
    if role or importance:
        parts.append(f"Role: {role}. Importance: {importance}.")
    return " ".join(parts)

def edge_prompt(edge: dict, *, reverse: bool = False) -> str:
    source = edge["target_kp_id"] if reverse else edge["source_kp_id"]
    target = edge["source_kp_id"] if reverse else edge["target_kp_id"]
    rationale = edge.get("keep_rationale", "")
    return (
        "Prerequisite relationship scoring. "
        f"Source prerequisite concept: {kp_text(source)} "
        f"Target concept to learn later: {kp_text(target)} "
        "Claim: a learner should understand the source concept before the target concept. "
        f"Existing rationale: {rationale}"
    )

print("Clean edges:", len(clean_edges))
print("KP catalog entries available:", len(kp_index))
print("Sample prompt:\n", edge_prompt(clean_edges[0])[:1000])

Clean edges: 93
KP catalog entries available: 0
Sample prompt:
 Prerequisite relationship scoring. Source prerequisite concept: Concept: 3d datasets and ai geometry task space. Target concept to learn later: Concept: multi view and voxel based 3d deep learning. Claim: a learner should understand the source concept before the target concept. Existing rationale: 3D datasets and AI+geometry task space provides prerequisite conceptual machinery, motivation, or implementation context needed to understand Multi-view and voxel-based 3D deep learning; the target specializes, applies, or operationalizes the source rather than merely co-occurring with it.


In [5]:
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("device=", device, "dtype=", dtype)
if device == "cuda":
    print("gpu=", torch.cuda.get_device_name(0))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    attn_implementation="sdpa",
).to(device)
model.eval()

def embed_texts(texts, batch_size=BATCH_SIZE):
    vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            output = model(**encoded)
            hidden = output.last_hidden_state
            mask = encoded["attention_mask"].unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            pooled = F.normalize(pooled.float(), p=2, dim=1)
            vectors.append(pooled.cpu())
    return torch.cat(vectors, dim=0)

positive_anchors = [
    "This is a strong prerequisite relationship: the source concept provides necessary conceptual machinery for understanding the target concept.",
    "The target concept depends on the source concept; learning the source first improves comprehension of the target.",
    "This edge represents a valid learning dependency, not merely topical co-occurrence."
]
negative_anchors = [
    "These concepts are only loosely related and do not form a prerequisite dependency.",
    "The source and target co-occur in the course, but one is not needed to understand the other.",
    "This edge is weak, redundant, reversed, or out of scope for a prerequisite graph."
]

anchor_vecs = embed_texts(positive_anchors + negative_anchors, batch_size=6)
positive_vec = F.normalize(anchor_vecs[:len(positive_anchors)].mean(dim=0, keepdim=True), p=2, dim=1)
negative_vec = F.normalize(anchor_vecs[len(positive_anchors):].mean(dim=0, keepdim=True), p=2, dim=1)
print("Model loaded and anchors embedded")

device= cuda dtype= torch.float16
gpu= Tesla T4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded and anchors embedded


In [6]:
forward_prompts = [edge_prompt(edge, reverse=False) for edge in clean_edges]
reverse_prompts = [edge_prompt(edge, reverse=True) for edge in clean_edges]

forward_vecs = embed_texts(forward_prompts)
reverse_vecs = embed_texts(reverse_prompts)

def anchor_score(vec):
    pos = F.cosine_similarity(vec, positive_vec).item()
    neg = F.cosine_similarity(vec, negative_vec).item()
    # Convert relative anchor preference to [0, 1]. This is a bootstrap scorer,
    # not supervised psychometric calibration.
    score = 1.0 / (1.0 + math.exp(-8.0 * (pos - neg)))
    return max(0.0, min(1.0, score)), pos, neg

scored_edges = []
for edge, fvec, rvec in zip(clean_edges, forward_vecs, reverse_vecs):
    edge_strength, forward_pos_sim, forward_neg_sim = anchor_score(fvec.unsqueeze(0))
    reverse_strength, reverse_pos_sim, reverse_neg_sim = anchor_score(rvec.unsqueeze(0))
    direction_margin = edge_strength - reverse_strength
    if edge_strength >= 0.5 and reverse_strength <= 0.3:
        modernbert_review_status = "not_required"
    elif edge_strength >= 0.3:
        modernbert_review_status = "optional"
    else:
        modernbert_review_status = "deferred"

    scored = dict(edge)
    scored.update({
        "edge_strength": round(edge_strength, 4),
        "bidirectional_score": round(reverse_strength, 4),
        "direction_margin": round(direction_margin, 4),
        "modernbert_model": MODEL_ID,
        "modernbert_scoring_method": "anchor_embedding_contrast_v1",
        "modernbert_review_status": modernbert_review_status,
        "modernbert_debug": {
            "forward_pos_sim": round(forward_pos_sim, 4),
            "forward_neg_sim": round(forward_neg_sim, 4),
            "reverse_pos_sim": round(reverse_pos_sim, 4),
            "reverse_neg_sim": round(reverse_neg_sim, 4),
        },
    })
    scored_edges.append(scored)

print("Scored edges:", len(scored_edges))
print("First scored edge:")
print(json.dumps(scored_edges[0], indent=2)[:1500])

Scored edges: 93
First scored edge:
{
  "source_kp_id": "kp_3d_datasets_and_ai_geometry_task_space",
  "target_kp_id": "kp_multi_view_and_voxel_based_3d_deep_learning",
  "edge_scope": "intra_course",
  "provenance": "llm_cross_check",
  "keep_confidence": "medium",
  "keep_rationale": "3D datasets and AI+geometry task space provides prerequisite conceptual machinery, motivation, or implementation context needed to understand Multi-view and voxel-based 3D deep learning; the target specializes, applies, or operationalizes the source rather than merely co-occurring with it.",
  "expected_directionality": "moderate",
  "review_status": "optional",
  "ready_for_modernbert": true,
  "edge_strength": 0.5446,
  "bidirectional_score": 0.5463,
  "direction_margin": -0.0017,
  "modernbert_model": "answerdotai/ModernBERT-base",
  "modernbert_scoring_method": "anchor_embedding_contrast_v1",
  "modernbert_review_status": "optional",
  "modernbert_debug": {
    "forward_pos_sim": 0.9377,
    "forwar

In [7]:
from collections import Counter

output = {
    "run_id": payload.get("run_id", "p5_cs224n_cs231n"),
    "stage_id": "modernbert_edge_scoring",
    "source_input_path": str(input_json_path),
    "model_id": MODEL_ID,
    "max_length": MAX_LENGTH,
    "scoring_method": "anchor_embedding_contrast_v1",
    "scored_edges": scored_edges,
    "score_summary": {
        "edge_count": len(scored_edges),
        "modernbert_review_status_distribution": dict(Counter(e["modernbert_review_status"] for e in scored_edges)),
        "edge_strength_min": min(e["edge_strength"] for e in scored_edges),
        "edge_strength_max": max(e["edge_strength"] for e in scored_edges),
        "edge_strength_mean": round(sum(e["edge_strength"] for e in scored_edges) / len(scored_edges), 4),
        "bidirectional_score_mean": round(sum(e["bidirectional_score"] for e in scored_edges) / len(scored_edges), 4),
    },
}

Path(OUTPUT_PATH).write_text(json.dumps(output, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(json.dumps(output["score_summary"], indent=2))
print("Wrote", OUTPUT_PATH)

{
  "edge_count": 93,
  "modernbert_review_status_distribution": {
    "optional": 93
  },
  "edge_strength_min": 0.5311,
  "edge_strength_max": 0.5528,
  "edge_strength_mean": 0.5417,
  "bidirectional_score_mean": 0.541
}
Wrote /content/modernbert_edge_scores.json


In [9]:
from google.colab import drive
import shutil
from pathlib import Path

drive.mount("/content/drive")

drive_output = Path("/content/drive/MyDrive/modernbert_edge_scores.json")
shutil.copyfile(OUTPUT_PATH, drive_output)

print(f"Saved output to Google Drive: {drive_output}")

Mounted at /content/drive
Saved output to Google Drive: /content/drive/MyDrive/modernbert_edge_scores.json
